# Getting started with entarchy

This notebook builds a small entarchy from scratch, stores some data on it and
queries it back. It uses the SQLite backend and a temporary directory, so it is
safe to run anywhere and leaves nothing behind.

The schema below is defined in this notebook for convenience. For real projects,
and especially for parallel processing, define entity types in an importable
module instead (see `02_parallel_analysis.ipynb`).

In [ ]:
import os
import shutil
import tempfile

import numpy as np

import entarchy
from entarchy.backend import SQLiteBackend

## Defining a hierarchy

An entarchy is a tree of entity types. Here: an `Animal` has `Recording`s, and a
recording has `Roi`s. Each type is a plain class; the hierarchy is declared by
registering child types.

In [ ]:
class Animal(entarchy.Entity):
    pass


class Recording(entarchy.Entity):
    pass


class Roi(entarchy.Entity):
    pass


Animal.add_child_entity_type(Recording)
Recording.add_child_entity_type(Roi)


class Demo(entarchy.Entarchy):
    _implementation_version = '0.1'
    _implementation_compat_version_list = ['0.1']
    _hierarchy_root_type = Animal

## Creating an entarchy

`create()` writes a configuration file and initialises the backend. The returned
object is ready to use.

In [ ]:
path = os.path.join(tempfile.mkdtemp(), 'demo_entarchy')

ent = Demo.create(path, SQLiteBackend(path, dbname='demo.db'))
ent

## Adding entities

Entities are created with an id and a parent. Inside a `with ent:` block, writes
are batched and committed together, which is much faster than committing each
attribute on its own.

Attributes are set like dictionary items. Scalars are stored in typed columns;
anything else (arrays, lists, dicts) is stored as a blob.

In [ ]:
rng = np.random.default_rng(0)

with ent:
    for animal_index in range(2):
        animal = Animal(ent, _id=f'animal_{animal_index}', _parent=ent.root)
        ent.add_new_entity(animal)
        animal['strain'] = 'wildtype' if animal_index == 0 else 'mutant'
        animal['age_dpf'] = 6 + animal_index

        for rec_index in range(2):
            recording = Recording(ent, _id=f'rec_{rec_index}', _parent=animal)
            ent.add_new_entity(recording)
            recording['imaging_rate'] = 10.0

            for roi_index in range(25):
                roi = Roi(ent, _id=f'roi_{roi_index}', _parent=recording)
                ent.add_new_entity(roi)
                roi['index'] = roi_index
                roi['quality'] = float(rng.random())
                roi['trace'] = rng.normal(size=200)

print(f'{len(ent.get(Animal))} animals, {len(ent.get(Recording))} recordings, '
      f'{len(ent.get(Roi))} rois')

## Collections

`ent.get(<type>)` returns a collection, which is a lazy query rather than a list
of objects. In a notebook it renders as a summary; nothing is loaded until you
ask for values.

In [ ]:
rois = ent.get(Roi)
rois

`preview()` loads the first few entities with their scalar attributes, which is
handy while exploring. Blob attributes are excluded unless requested.

In [ ]:
rois.preview(5)

## Querying

Filters are written as strings. Comparisons use `==`, `!=`, `<`, `<=`, `>`, `>=`,
`IN (...)` and `EXIST(...)`; they combine with `AND`, `OR`, `XOR` and `NOT` using
the usual precedence.

In [ ]:
good = ent.get(Roi, 'quality > 0.8')
print(f'{len(good)} good rois')

# Attributes of ancestors are addressed by type name
wildtype = ent.get(Roi, '[Animal]strain == "wildtype"')
print(f'{len(wildtype)} rois from wildtype animals')

# ... or relative to the current level, one "../" per step up
same = ent.get(Roi, '../../strain == "wildtype"')
print(f'{len(same)} rois via relative addressing')

Collections support set operations, so queries can be composed.

In [ ]:
combined = good & wildtype
print(f'{len(combined)} good wildtype rois')
print(f'{len(good - wildtype)} good rois that are not wildtype')

## Reading and writing in bulk

`dataframe_of()` returns a pandas DataFrame, including ancestor attributes. Writing
a column back to a collection stores it on every entity in one statement.

In [ ]:
df = ent.get(Roi).dataframe_of(['index', 'quality', '[Animal]strain'])
df.head()

In [ ]:
# Derived values can be written straight back
rois = ent.get(Roi)
rois['quality_rank'] = rois['quality'].rank()

ent.get(Roi).dataframe_of(['quality', 'quality_rank']).head()

## Single entities

Indexing a collection gives an entity. Reading an attribute loads it on demand and
caches it.

In [ ]:
roi = ent.get(Roi, 'index == 0')[0]
roi

In [ ]:
trace = roi['trace']
print(type(trace), trace.shape, trace[:5])

## Cleaning up

In [ ]:
ent.backend.close()
shutil.rmtree(os.path.dirname(path), ignore_errors=True)
print('done')